# Intel Image Classification - PyTorch Version
This notebook contains the PyTorch implementation for classifying images into 6 categories: buildings, forest, glacier, mountain, sea, and street.
The model is trained on the Intel Dataset from Kaggle competition.

In [ ]:
# Download the Intel Image Classification dataset using kagglehub
import kagglehub

# Download latest version
path = kagglehub.dataset_download("puneet6060/intel-image-classification")

print("Path to dataset files:", path)

In [ ]:
# Import necessary libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os
from pathlib import Path
from tqdm import tqdm

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

First, let's define the paths for our training and testing data. The `path` variable was already created from the dataset download.

In [ ]:
# Define paths for training and testing directories
train_dir = os.path.join(path, 'seg_train', 'seg_train')
test_dir = os.path.join(path, 'seg_test', 'seg_test')

print(f"Training data directory: {train_dir}")
print(f"Testing data directory: {test_dir}")

# Check if directories exist and list classes
if os.path.exists(train_dir):
    classes = sorted(os.listdir(train_dir))
    print(f"\nClasses found: {classes}")
    print(f"Number of classes: {len(classes)}")

Now we'll define a custom PyTorch Dataset that loads images from directories and handles the image preprocessing.

In [ ]:
# Define custom Dataset class for Intel Image Classification
class IntelImageDataset(Dataset):
    def __init__(self, directory, class_names, transform=None):
        """
        Args:
            directory (str): Path to the dataset directory
            class_names (list): List of class names in order
            transform (callable, optional): Optional transform to be applied on images
        """
        self.directory = directory
        self.class_names = class_names
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(class_names)}
        self.transform = transform
        self.images = []
        self.labels = []
        
        # Load image paths and labels from directory structure
        for class_name in class_names:
            class_dir = os.path.join(directory, class_name)
            if os.path.exists(class_dir):
                for img_file in os.listdir(class_dir):
                    img_path = os.path.join(class_dir, img_file)
                    if os.path.isfile(img_path):
                        try:
                            self.images.append(img_path)
                            self.labels.append(self.class_to_idx[class_name])
                        except:
                            continue
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_path = self.images[idx]
        label = self.labels[idx]
        
        # Read image
        try:
            image = cv2.imread(img_path)
            if image is not None:
                image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
                image = cv2.resize(image, (150, 150))
                
                if self.transform:
                    image = self.transform(image)
                else:
                    # Default: convert to tensor and normalize
                    image = torch.tensor(image, dtype=torch.float32) / 255.0
                    image = image.permute(2, 0, 1)  # HWC to CHW
            else:
                # Return a zero tensor if image reading fails
                image = torch.zeros(3, 150, 150)
        except:
            image = torch.zeros(3, 150, 150)
        
        return image, label


# Define class names
class_names = ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
print(f"Classes: {class_names}")
print(f"Number of classes: {len(class_names)}")

In [ ]:
# Define data transforms
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Create training dataset (with manual train/validation split)
train_dataset = IntelImageDataset(train_dir, class_names, transform=transform)

print(f"Train dataset size: {len(train_dataset)}")

# Create a validation split from training data
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_dataset_split, val_dataset_split = torch.utils.data.random_split(
    train_dataset, 
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

print(f"Train split size: {len(train_dataset_split)}")
print(f"Validation split size: {len(val_dataset_split)}")

# Create test dataset
test_dataset = IntelImageDataset(test_dir, class_names, transform=transform)
print(f"Test dataset size: {len(test_dataset)}")

In [ ]:
# Create DataLoaders for batch processing
batch_size = 32
train_loader = DataLoader(train_dataset_split, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset_split, batch_size=batch_size, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

# Verify batch shapes
for images, labels in train_loader:
    print(f"\nShape of a batch of training images: {images.shape}")
    print(f"Shape of a batch of training labels: {labels.shape}")
    break

Define the CNN model with 4 convolutional blocks followed by dense layers.

In [ ]:
# Define the CNN model for 6-class classification
class IntelCNN(nn.Module):
    def __init__(self, num_classes=6):
        super(IntelCNN, self).__init__()
        
        # First convolutional block
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=0)
        self.bn1 = nn.BatchNorm2d(32)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Second convolutional block
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=0)
        self.bn2 = nn.BatchNorm2d(64)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Third convolutional block
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=0)
        self.bn3 = nn.BatchNorm2d(128)
        self.relu3 = nn.ReLU()
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Fourth convolutional block
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=0)
        self.bn4 = nn.BatchNorm2d(256)
        self.relu4 = nn.ReLU()
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Flatten layer
        self.flatten = nn.Flatten()
        
        # Fully connected layers
        self.fc1 = nn.Linear(256 * 7 * 7, 256)  # Adjusted based on input size
        self.dropout1 = nn.Dropout(0.2)
        self.relu_fc1 = nn.ReLU()
        
        self.fc2 = nn.Linear(256, 128)
        self.dropout2 = nn.Dropout(0.2)
        self.relu_fc2 = nn.ReLU()
        
        self.fc3 = nn.Linear(128, 64)
        self.dropout3 = nn.Dropout(0.2)
        self.relu_fc3 = nn.ReLU()
        
        # Output layer for multi-class classification
        self.fc4 = nn.Linear(64, num_classes)
    
    def forward(self, x):
        # First block
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu1(x)
        x = self.pool1(x)
        
        # Second block
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu2(x)
        x = self.pool2(x)
        
        # Third block
        x = self.conv3(x)
        x = self.bn3(x)
        x = self.relu3(x)
        x = self.pool3(x)
        
        # Fourth block
        x = self.conv4(x)
        x = self.bn4(x)
        x = self.relu4(x)
        x = self.pool4(x)
        
        # Fully connected layers
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.dropout1(x)
        x = self.relu_fc1(x)
        
        x = self.fc2(x)
        x = self.dropout2(x)
        x = self.relu_fc2(x)
        
        x = self.fc3(x)
        x = self.dropout3(x)
        x = self.relu_fc3(x)
        
        x = self.fc4(x)
        
        return x

# Initialize the model
model = IntelCNN(num_classes=len(class_names)).to(device)
print("Model initialized successfully!")
print(model)

In [ ]:
# Print model summary with parameter count
def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

total_params, trainable_params = count_parameters(model)
print(f"\nModel Summary:")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()  # For multi-class classification
optimizer = optim.Adam(model.parameters())

print("Loss function: CrossEntropyLoss")
print("Optimizer: Adam")

In [ ]:
# Training function
def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in tqdm(train_loader, desc="Training"):
        images = images.to(device)
        labels = labels.to(device)
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        # Calculate accuracy
        _, predicted = torch.max(outputs.data, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
    
    avg_loss = total_loss / len(train_loader)
    accuracy = 100 * correct / total
    
    return avg_loss, accuracy


# Validation function
def validate(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc="Validating"):
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            
            _, predicted = torch.max(outputs.data, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    
    avg_loss = total_loss / len(val_loader)
    accuracy = 100 * correct / total
    
    return avg_loss, accuracy


print("Training and validation functions defined!")

In [ ]:
# Train the model
num_epochs = 10
history = {
    'train_loss': [],
    'val_loss': [],
    'train_accuracy': [],
    'val_accuracy': []
}

for epoch in range(num_epochs):
    print(f"\nEpoch [{epoch+1}/{num_epochs}]")
    
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_accuracy'].append(train_acc)
    history['val_accuracy'].append(val_acc)
    
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")

print("\nTraining completed!")

In [ ]:
# Plot training history
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history['train_accuracy'], label='Train Accuracy')
plt.plot(history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title('Training and Validation Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Function to predict on a single image
def predict_image(model, image_path, class_names, device):
    """
    Predict the class of an image
    Args:
        model: Trained PyTorch model
        image_path: Path to the image file
        class_names: List of class names
        device: Device to run inference on (CPU or GPU)
    """
    try:
        # Read and preprocess image
        test_img = cv2.imread(image_path)
        if test_img is None:
            print(f"Could not read image from {image_path}")
            return None
        
        test_img_rgb = cv2.cvtColor(test_img, cv2.COLOR_BGR2RGB)
        test_img_resized = cv2.resize(test_img_rgb, (150, 150))
        
        # Convert to tensor
        test_tensor = torch.tensor(test_img_resized, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0) / 255.0
        test_tensor = test_tensor.to(device)
        
        # Make prediction
        model.eval()
        with torch.no_grad():
            outputs = model(test_tensor)
            _, predicted = torch.max(outputs.data, 1)
            probabilities = torch.nn.functional.softmax(outputs, dim=1)
        
        pred_class = class_names[predicted.item()]
        confidence = probabilities[0][predicted].item()
        
        return pred_class, confidence, test_img_rgb
    except Exception as e:
        print(f"Error: {e}")
        return None


print("Prediction function defined!")

In [ ]:
# Test the model on sample images
test_images = [
    '/content/images-2.jpeg',
    '/content/forest-path-v3-13197528.jpg.webp',
    '/content/GlacierBlues_Header3.jpg.webp',
    '/content/mt-everest.jpg.avif',
    '/content/beautiful-beach-tropical-beach-background-as-summer-landscape-white-sand-and-calm-sea-for-beach-banner-perfect-beach-scene-vacation-and-summer-holiday-concept-boost-up-color-process-photo.jpg',
    '/content/istockphoto-1628010210-612x612.jpg'
]

print("Testing model on sample images...\n")
for img_path in test_images:
    result = predict_image(model, img_path, class_names, device)
    if result:
        pred_class, confidence, img = result
        print(f"Image: {os.path.basename(img_path)}")
        print(f"Predicted class: {pred_class}")
        print(f"Confidence: {confidence:.4f}")
        print()

In [ ]:
# Visualize predictions on sample test images
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, img_path in enumerate(test_images[:6]):
    result = predict_image(model, img_path, class_names, device)
    if result:
        pred_class, confidence, img = result
        axes[idx].imshow(img)
        axes[idx].set_title(f"{pred_class} ({confidence:.2%})")
        axes[idx].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Save the trained model
model_save_path = 'intel_image_classifier_pytorch.pth'
torch.save(model.state_dict(), model_save_path)
print(f"Model state dict saved to {model_save_path}")

# Also save the entire model object
model_full_path = 'intel_image_classifier_pytorch_full.pt'
torch.save(model, model_full_path)
print(f"Full model saved to {model_full_path}")

# Save class names for reference
import json
with open('intel_class_names.json', 'w') as f:
    json.dump(class_names, f)
print(f"Class names saved to intel_class_names.json")

In [ ]:
# Optional: Load the model for future use
def load_model(model_path, num_classes=6):
    """
    Load a saved PyTorch model
    """
    model = IntelCNN(num_classes=num_classes)
    model.load_state_dict(torch.load(model_path))
    model = model.to(device)
    return model

# Example usage:
# loaded_model = load_model(model_save_path)
# loaded_model.eval()